# 00 · Features compartidas

Construye una sola vez todo lo que usan las notebooks de modelos y lo guarda en `cache/`:

1. **Features de fila** y **agregados no supervisados** por vendedor/categoría (`features.py`), calculados sobre train+test juntos porque no usan la etiqueta.
2. **Features de nivel 1**, que sí usan la etiqueta y por eso se calculan siempre *out-of-fold*:
   - target encoding suavizado de vendedor, categoría, ciudad, garantía y vendedor×categoría, y de combinaciones
     vendedor × atributo (tipo de publicación, stock 1, garantía, envío, rango de precio, palabras nuevo/usado);
   - vecinos de título dentro del mismo vendedor: etiqueta de las publicaciones más parecidas de ese vendedor;
   - score de una regresión logística sobre TF-IDF (palabras + caracteres) del título y la garantía;
   - score de una regresión logística sobre embeddings del título (`multilingual-e5-small`);
   - proporción de vecinos "new" entre los k títulos más parecidos (kNN sobre embeddings).

Se generan dos versiones: una por cada fold de validación (ajustada solo con la parte de entrenamiento del fold, para que la validación no tenga fuga de información) y una final (ajustada con todo train, aplicada a test).

In [1]:
import os, numpy as np, pandas as pd, scipy.sparse as sp
import features as F
os.makedirs(F.CACHE, exist_ok=True)

train, test = F.load("train_data.jsonlines"), F.load("test_data.jsonlines")
y = (train[F.TARGET] == "new").astype(int).values
X_train, X_test = F.build(train, test)
np.save(f"{F.CACHE}/y.npy", y)
test[["id"]].to_csv(f"{F.CACHE}/test_ids.csv", index=False)
X_train.to_parquet(f"{F.CACHE}/Xtr.parquet"); X_test.to_parquet(f"{F.CACHE}/Xte.parquet")
print(X_train.shape, X_test.shape, "proporción new:", y.mean().round(3))

(70000, 173) (30000, 173) proporción new: 0.537


## Texto: TF-IDF
El vocabulario se ajusta con train+test (no usa etiquetas).

In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer
txt = np.concatenate([F.text_field(train), F.text_field(test)])
W = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True).fit_transform(txt)
C = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=3, sublinear_tf=True, max_features=400_000).fit_transform(txt)
T = sp.hstack([W, C]).tocsr()
T_train, T_test = T[:len(train)], T[len(train):]
T.shape

(100000, 254377)

## Texto: embeddings
Modelo preentrenado multilingüe. En CPU tarda ~20 min, por eso se cachea.

In [3]:
emb_path = f"{F.CACHE}/emb_e5small.npy"
if os.path.exists(emb_path):
    E = np.load(emb_path)
else:
    from sentence_transformers import SentenceTransformer
    st = SentenceTransformer("intfloat/multilingual-e5-small", device="cpu"); st.max_seq_length = 48
    titles = ["query: " + t for t in pd.concat([train["title"], test["title"]]).fillna("")]
    E = st.encode(titles, batch_size=128, normalize_embeddings=True, show_progress_bar=False).astype("float32")
    np.save(emb_path, E)
E_train, E_test = E[:len(train)], E[len(train):]
E.shape

(100000, 384)

## Features de nivel 1 por fold y finales

In [4]:
for f, (a, b) in enumerate(F.cv_folds(y)):
    A, B = F.level1(X_train.iloc[a], y[a], X_train.iloc[b], T_train[a], T_train[b], E_train[a], E_train[b])
    A.to_parquet(f"{F.CACHE}/fold{f}_tr.parquet"); B.to_parquet(f"{F.CACHE}/fold{f}_va.parquet")
    print(f"fold {f}: {A.shape} | accuracy solo TF-IDF {((B.txt_tfidf > 0) == y[b]).mean():.4f} | solo embeddings {((B.txt_emb > 0) == y[b]).mean():.4f} | solo kNN {((B.knn_p20 > .5) == y[b]).mean():.4f}")

A, B = F.level1(X_train, y, X_test, T_train, T_test, E_train, E_test)
A.to_parquet(f"{F.CACHE}/full_tr.parquet"); B.to_parquet(f"{F.CACHE}/full_te.parquet")
A.shape, B.shape

fold 0: (56000, 197) | accuracy solo TF-IDF 0.8419 | solo embeddings 0.7666 | solo kNN 0.8141


fold 1: (56000, 197) | accuracy solo TF-IDF 0.8325 | solo embeddings 0.7588 | solo kNN 0.8059


fold 2: (56000, 197) | accuracy solo TF-IDF 0.8361 | solo embeddings 0.7654 | solo kNN 0.8098


fold 3: (56000, 197) | accuracy solo TF-IDF 0.8384 | solo embeddings 0.7623 | solo kNN 0.8073


fold 4: (56000, 197) | accuracy solo TF-IDF 0.8329 | solo embeddings 0.7557 | solo kNN 0.8076


((70000, 197), (30000, 197))